 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : John Ryan         <br>
Student Number : L00007202         <br>
Due Date       : 12th May 2026     <br>
Assignment     : CA2               <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [2]:
# pip installs
# pip install -r requirements.txt

## Imports

In [1]:
# imports
import pandas as pd

# Support Functions

In [4]:
# code here

# NLP

## Define the Corpus

In [2]:
# import the required libraries
import pymupdf
import pytesseract
import os
from PIL import Image
import cv2
import numpy as np
import io
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe" #path to the tesseract program on my laptop

# define variable with the strings of the file names included in the corpus
document_paths = [
    "schoolpros1.jpg",
    "schoolpros2.jpg",
    "schoolpros3.jpg",
    "schoolpros4.jpg",
    "schoolpros5.jpg",
    "schoolpros6.jpg",
    "schoolpros7.jpg",
    "schoolpros8.jpg",
    "schoolpros9.jpg",
    "schoolpros10.jpg",
    "schoolpros11.jpg",
    "schoolpros12.jpg",
    "schoolpros13.jpg",
    "schoolpros14.jpg"
    # more docs maybe...nope!
]
# set up the function that will extract the text from the JPEG files
def extract_text_from_jpg(path): #define the function to read a JPEG
    img = Image.open(path) #open JPEG at the path using PIL
    #img = img.convert("L") # "L" mode = greyscale.Converting to greyscale to improve OCR accuracy

    #using OpenCV to preprocess to improve OCR accuracy
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)  # convert to greyscale
    img_cv = cv2.threshold(img_cv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]  # apply Otsu's thresholding to binarise the image
    img = Image.fromarray(img_cv)  # convert back to PIL image for pytesseract
        
    text = pytesseract.image_to_string(img) # run OCR on the preprocessed image using tesseract to extract text from the image and add it to the string
    return text # returns full extract

# go through the JPGs listed in 'document_paths', extract the text and store it in a new dictionary
corpus_raw = {} #name of new dictionary
for path in document_paths: #loop through the docs listed in' document_paths'
    doc_name = os.path.basename(path)  #the filename will be the key
    corpus_raw[doc_name] = extract_text_from_jpg(path) #extracts the text from JPEGs and add to dictionary
    print(f"Loaded: {doc_name} — {len(corpus_raw[doc_name])} characters") #print out status with char count

print(f"\nTotal documents in corpus: {len(corpus_raw)}") #number of docs in the corpus/dictionary

Loaded: schoolpros1.jpg — 53 characters
Loaded: schoolpros2.jpg — 26 characters
Loaded: schoolpros3.jpg — 258 characters
Loaded: schoolpros4.jpg — 1470 characters
Loaded: schoolpros5.jpg — 854 characters
Loaded: schoolpros6.jpg — 1010 characters
Loaded: schoolpros7.jpg — 860 characters
Loaded: schoolpros8.jpg — 12 characters
Loaded: schoolpros9.jpg — 855 characters
Loaded: schoolpros10.jpg — 1389 characters
Loaded: schoolpros11.jpg — 1180 characters
Loaded: schoolpros12.jpg — 1039 characters
Loaded: schoolpros13.jpg — 312 characters
Loaded: schoolpros14.jpg — 154 characters

Total documents in corpus: 14


## Document Text Processing

In [6]:
#!python -m spacy download en_core_web_sm #RAN ONCE ON MAY 3RD, SHOULDNT NEED TO RUN AGAIN

In [3]:
# import modules etc
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import spacy

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

nlp = spacy.load("en_core_web_sm") #load the small English NLP model from spacy
stop_words = set(stopwords.words('english')) #load nltk stopwords
stemmer = PorterStemmer() #load Porter's stemmer

# set up the function to NLP pre-proc, i.e. tokenisation, remove stopswords, stem and lemma the text
def process_document(text): #define the function for pre-proc
    tokens = word_tokenize(text) #tokenise the text breaking into words and punc marks
    tokens = [t.lower() for t in tokens if t.isalpha()] #set to lowercase and remove numbers
    tokens = [t for t in tokens if t not in stop_words] #remove stopwords from tokens
    stemmed = [stemmer.stem(t) for t in tokens] #stem the tokens (porter)
    doc_spacy = nlp(" ".join(tokens)) #rejoin the tokens for lemmatisation
    lemmatised = [token.lemma_.lower() for token in doc_spacy if token.is_alpha] #lemmatise the tokens
    
    return {
        "tokens"    : tokens, #lowercase with stopwords removed
        "stemmed"   : stemmed, #stemmed versions
        "lemmatised": lemmatised, #lemmatised verstions using spacy
        "processed_text": " ".join(lemmatised) #single string of lemmatised words
    }

#Run the process_document function on the docs in the corpus
corpus_processed = {} #create a new dictionary which will hold the processed data
for doc_name, raw_text in corpus_raw.items(): #loop through each doc in the corpus
    print(f"Processing: {doc_name}") #display name of docs processed
    corpus_processed[doc_name] = process_document(raw_text) #apply the pre-proc function and send the output to the new dictionary

print("\nAll documents processed like!") #status message

# ── 6. Build processed text list for TF-IDF ──────────────────────────────────
tfidf_corpus = [corpus_processed[doc]["processed_text"] for doc in corpus_processed] #build list of processed text for each doc, i.e. list with 3 parts
doc_names    = list(corpus_processed.keys()) #save list of doc names / keys

print(f"\nCorpus ready for TF-IDF: {len(tfidf_corpus)} documents") #status message

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jryan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Processing: schoolpros1.jpg
Processing: schoolpros2.jpg
Processing: schoolpros3.jpg
Processing: schoolpros4.jpg
Processing: schoolpros5.jpg
Processing: schoolpros6.jpg
Processing: schoolpros7.jpg
Processing: schoolpros8.jpg
Processing: schoolpros9.jpg
Processing: schoolpros10.jpg
Processing: schoolpros11.jpg
Processing: schoolpros12.jpg
Processing: schoolpros13.jpg
Processing: schoolpros14.jpg

All documents processed like!

Corpus ready for TF-IDF: 14 documents


## Tokenisation

### Summary Token Info for the Documents

In [5]:
for doc_name, data in corpus_processed.items(): #loop through the docs in the corpus
    print(f"\n=== {doc_name} ===") #print doc name
    print(f"Token count: {len(data['tokens'])}") #print tokens qty
    print(f"Unique tokens: {len(set(data['tokens']))}") #print unique tokens qty
    print(f"First 5 tokens: {data['tokens'][:5]}") #as example, show first 5 tokens 


=== schoolpros1.jpg ===
Token count: 2
Unique tokens: 2
First 5 tokens: ['wisdom', 'courage']

=== schoolpros2.jpg ===
Token count: 2
Unique tokens: 2
First 5 tokens: ['year', 'student']

=== schoolpros3.jpg ===
Token count: 25
Unique tokens: 23
First 5 tokens: ['contents', 'page', 'message', 'principal', 'mission']

=== schoolpros4.jpg ===
Token count: 142
Unique tokens: 123
First 5 tokens: ['message', 'principal', 'yvonne', 'lucey', 'principal']

=== schoolpros5.jpg ===
Token count: 69
Unique tokens: 64
First 5 tokens: ['mission', 'statement', 'regina', 'mundi', 'college']

=== schoolpros6.jpg ===
Token count: 84
Unique tokens: 67
First 5 tokens: ['oo', 'nti', 'nii', 'ot', 'loo']

=== schoolpros7.jpg ===
Token count: 74
Unique tokens: 65
First 5 tokens: ['teaching', 'learning', 'highest', 'quality', 'teaching']

=== schoolpros8.jpg ===
Token count: 2
Unique tokens: 2
First 5 tokens: ['ss', 'pos']

=== schoolpros9.jpg ===
Token count: 86
Unique tokens: 56
First 5 tokens: ['junior', '

In [6]:
from collections import Counter #import mod to count words

for doc_name, data in corpus_processed.items(): #loop through docs in the corpus
    freq = Counter(data['tokens']) 
    common = freq.most_common(5) #get the 5 most plentiful lemmatised words
    print(f"\nTop words in {doc_name}:") #print the output with a doc name heading
    for word, count in common:
        print(f"  {word}: {count}") #print the lemma and its frequency count


Top words in schoolpros1.jpg:
  wisdom: 1
  courage: 1

Top words in schoolpros2.jpg:
  year: 1
  student: 1

Top words in schoolpros3.jpg:
  learning: 2
  cycle: 2
  contents: 1
  page: 1
  message: 1

Top words in schoolpros4.jpg:
  school: 4
  principal: 3
  students: 3
  regina: 3
  mundi: 3

Top words in schoolpros5.jpg:
  school: 3
  aims: 2
  student: 2
  students: 2
  mission: 1

Top words in schoolpros6.jpg:
  community: 4
  school: 3
  service: 3
  regina: 2
  mundi: 2

Top words in schoolpros7.jpg:
  learning: 3
  teaching: 2
  academic: 2
  expertise: 2
  students: 2

Top words in schoolpros8.jpg:
  ss: 1
  pos: 1

Top words in schoolpros9.jpg:
  cycle: 4
  junior: 3
  mathematics: 3
  following: 2
  subjects: 2

Top words in schoolpros10.jpg:
  students: 6
  learning: 5
  activities: 5
  enrichment: 4
  experience: 2

Top words in schoolpros11.jpg:
  school: 5
  pastoral: 3
  care: 3
  students: 3
  well: 3

Top words in schoolpros12.jpg:
  school: 10
  students: 7
  must

## Unique Words and Keywords
After Stopwords are Removed

Generate a bigger sample of 30 tokens per doc - IS THIS ADDING ANY VALUE???

In [7]:
for doc_name, data in corpus_processed.items(): #loop through docs in corpus
    print(f"\n=== {doc_name} ===")
    #print(f"Token count after stopword removal: {len(data['tokens'])}")
    print(f"Sample tokens: {data['tokens'][:10]}") #print a longer list of tokens for each doc as a sample


=== schoolpros1.jpg ===
Sample tokens: ['wisdom', 'courage']

=== schoolpros2.jpg ===
Sample tokens: ['year', 'student']

=== schoolpros3.jpg ===
Sample tokens: ['contents', 'page', 'message', 'principal', 'mission', 'statement', 'philosophy', 'ethos', 'teaching', 'learning']

=== schoolpros4.jpg ===
Sample tokens: ['message', 'principal', 'yvonne', 'lucey', 'principal', 'immensely', 'proud', 'lead', 'exceptional', 'school']

=== schoolpros5.jpg ===
Sample tokens: ['mission', 'statement', 'regina', 'mundi', 'college', 'voluntary', 'secondary', 'school', 'founded', 'miss']

=== schoolpros6.jpg ===
Sample tokens: ['oo', 'nti', 'nii', 'ot', 'loo', 'philosophy', 'ethos', 'founding', 'regina', 'mundi']

=== schoolpros7.jpg ===
Sample tokens: ['teaching', 'learning', 'highest', 'quality', 'teaching', 'learning', 'lies', 'heart', 'success', 'outstanding']

=== schoolpros8.jpg ===
Sample tokens: ['ss', 'pos']

=== schoolpros9.jpg ===
Sample tokens: ['junior', 'cycle', 'following', 'subjects',

Identify the Words Unique to each Document

In [8]:
doc_words = {name: set(data['tokens']) for name, data in corpus_processed.items()}#creates a new dict comprising sets of tokens

for name, words in doc_words.items(): #loop through the docs to get meaningful words in each doc
    others = set().union(*(doc_words[n] for n in doc_words if n != name)) #create 'others' to be the combo of the docs not being analysed
    unique = words - others # creates unique list by taking words in current doc from combo
    print(f"\nUnique meaningful words in {name}:") 
    print(sorted(unique)[:5]) #prints 5 unique words per doc ordered alphabetically


Unique meaningful words in schoolpros1.jpg:
[]

Unique meaningful words in schoolpros2.jpg:
[]

Unique meaningful words in schoolpros3.jpg:
['contents', 'page']

Unique meaningful words in schoolpros4.jpg:
['ach', 'agus', 'also', 'bualadh', 'carved']

Unique meaningful words in schoolpros5.jpg:
['accommodate', 'based', 'become', 'beliefs', 'board']

Unique meaningful words in schoolpros6.jpg:
['balance', 'begins', 'civic', 'concept', 'concern']

Unique meaningful words in schoolpros7.jpg:
['according', 'across', 'approach', 'attainment', 'classes']

Unique meaningful words in schoolpros8.jpg:
['pos', 'ss']

Unique meaningful words in schoolpros9.jpg:
['accounting', 'afforded', 'agricultural', 'applied', 'biology']

Unique meaningful words in schoolpros10.jpg:
['accordingly', 'adventure', 'artistic', 'assist', 'audiences']

Unique meaningful words in schoolpros11.jpg:
['able', 'air', 'allow', 'allowing', 'app']

Unique meaningful words in schoolpros12.jpg:
['adhere', 'ae', 'always', 'a

Identify the Keywords per Document - same, DOES THIS ADD ANY VALUE?

In [9]:
all_tokens = [] #a new list
for data in corpus_processed.values(): #looking in the corpus
    all_tokens.extend(data['tokens']) #combine tokens with stopword already removed

global_freq = Counter(all_tokens) #quantify the frequency of the tokens across all docs

for doc_name, data in corpus_processed.items(): #loop through
    doc_freq = Counter(data['tokens']) #quantify the frequency of the tokens across per doc
    keywords = [
        w for w in doc_freq
        if doc_freq[w] > 2 and global_freq[w] < 5 #keywords become those that appear >2 in a doc but <5 across all docs
    ]
    keywords_sorted = sorted(
    keywords,
    key=lambda w: doc_freq[w],
    reverse=True
    )
    print(f"\nKeywords for {doc_name} by frequency: {keywords_sorted[:5]}") #print the top 5 of each per doc



Keywords for schoolpros1.jpg by frequency: []

Keywords for schoolpros2.jpg by frequency: []

Keywords for schoolpros3.jpg by frequency: []

Keywords for schoolpros4.jpg by frequency: ['society']

Keywords for schoolpros5.jpg by frequency: []

Keywords for schoolpros6.jpg by frequency: ['service']

Keywords for schoolpros7.jpg by frequency: []

Keywords for schoolpros8.jpg by frequency: []

Keywords for schoolpros9.jpg by frequency: ['junior', 'mathematics']

Keywords for schoolpros10.jpg by frequency: []

Keywords for schoolpros11.jpg by frequency: ['pastoral', 'care', 'class', 'regular']

Keywords for schoolpros12.jpg by frequency: ['rules']

Keywords for schoolpros13.jpg by frequency: []

Keywords for schoolpros14.jpg by frequency: []


Lexical Diversity per Document - same, DOES THIS ADD ANY VALUE?

In [10]:
for doc_name, data in corpus_processed.items(): #loop through the corpus
    tokens = data['tokens'] #get the tokens
    unique = len(set(tokens)) #unique ones
    total = len(tokens)
    print(f"\n=== {doc_name} ===")
    if total == 0: #provision for images if no tokens found
        print ("Lexical diversity: N/A (no tokens found)") #if none, just print this msg
    else:
        print(f"Lexical diversity: {unique/total*100:.2f}%") #divide one by the other and print result as %



=== schoolpros1.jpg ===
Lexical diversity: 100.00%

=== schoolpros2.jpg ===
Lexical diversity: 100.00%

=== schoolpros3.jpg ===
Lexical diversity: 92.00%

=== schoolpros4.jpg ===
Lexical diversity: 86.62%

=== schoolpros5.jpg ===
Lexical diversity: 92.75%

=== schoolpros6.jpg ===
Lexical diversity: 79.76%

=== schoolpros7.jpg ===
Lexical diversity: 87.84%

=== schoolpros8.jpg ===
Lexical diversity: 100.00%

=== schoolpros9.jpg ===
Lexical diversity: 65.12%

=== schoolpros10.jpg ===
Lexical diversity: 78.23%

=== schoolpros11.jpg ===
Lexical diversity: 74.56%

=== schoolpros12.jpg ===
Lexical diversity: 68.00%

=== schoolpros13.jpg ===
Lexical diversity: 96.30%

=== schoolpros14.jpg ===
Lexical diversity: 94.12%


## Stemming

In [11]:
#from collections import Counter

for doc_name, data in corpus_processed.items(): #loop through
    stems = data['stemmed'] #variable stems
    total = len(stems) #totals stems
    unique = len(set(stems)) #unique stems
    print(f"\n=== {doc_name} ===")
    print(f"Total stems: {total}")
    print(f"Unique stems: {unique}")
    if total == 0: #provision for images if no stems found
        print ("Stemming compression: N/A (no stems found)") #just print this msg
    else:
        print(f"Stemming compression: {(1 - unique/total)*100:.2f}%") #unique as % of total stems



=== schoolpros1.jpg ===
Total stems: 2
Unique stems: 2
Stemming compression: 0.00%

=== schoolpros2.jpg ===
Total stems: 2
Unique stems: 2
Stemming compression: 0.00%

=== schoolpros3.jpg ===
Total stems: 25
Unique stems: 23
Stemming compression: 8.00%

=== schoolpros4.jpg ===
Total stems: 142
Unique stems: 119
Stemming compression: 16.20%

=== schoolpros5.jpg ===
Total stems: 69
Unique stems: 61
Stemming compression: 11.59%

=== schoolpros6.jpg ===
Total stems: 84
Unique stems: 66
Stemming compression: 21.43%

=== schoolpros7.jpg ===
Total stems: 74
Unique stems: 62
Stemming compression: 16.22%

=== schoolpros8.jpg ===
Total stems: 2
Unique stems: 2
Stemming compression: 0.00%

=== schoolpros9.jpg ===
Total stems: 86
Unique stems: 53
Stemming compression: 38.37%

=== schoolpros10.jpg ===
Total stems: 124
Unique stems: 87
Stemming compression: 29.84%

=== schoolpros11.jpg ===
Total stems: 114
Unique stems: 80
Stemming compression: 29.82%

=== schoolpros12.jpg ===
Total stems: 100
Uniq

Common Stems in the Documents- same, DOES THIS ADD ANY VALUE?

In [12]:
for doc_name, data in corpus_processed.items(): #loop through
    freq = Counter(data['stemmed']) #quantify the stems
    common = freq.most_common(5) #top 5
    print(f"\nTop stems in {doc_name}:") #print results
    for stem, count in common:
        print(f"  {stem}: {count}")



Top stems in schoolpros1.jpg:
  wisdom: 1
  courag: 1

Top stems in schoolpros2.jpg:
  year: 1
  student: 1

Top stems in schoolpros3.jpg:
  learn: 2
  cycl: 2
  content: 1
  page: 1
  messag: 1

Top stems in schoolpros4.jpg:
  school: 4
  princip: 3
  student: 3
  regina: 3
  mundi: 3

Top stems in schoolpros5.jpg:
  student: 4
  school: 3
  aim: 2
  educ: 2
  member: 2

Top stems in schoolpros6.jpg:
  commun: 4
  school: 3
  student: 3
  servic: 3
  regina: 2

Top stems in schoolpros7.jpg:
  learn: 3
  student: 3
  class: 3
  teach: 2
  academ: 2

Top stems in schoolpros8.jpg:
  ss: 1
  po: 1

Top stems in schoolpros9.jpg:
  cycl: 4
  junior: 3
  follow: 3
  mathemat: 3
  subject: 2

Top stems in schoolpros10.jpg:
  student: 7
  learn: 5
  activ: 5
  enrich: 4
  experi: 4

Top stems in schoolpros11.jpg:
  school: 5
  student: 5
  pastor: 3
  care: 3
  well: 3

Top stems in schoolpros12.jpg:
  school: 10
  student: 7
  must: 7
  rule: 3
  phone: 3

Top stems in schoolpros13.jpg:
  ma

## Lemmatisation

Lemma Overview per Document - - same, DOES THIS ADD ANY VALUE?

In [13]:
#from collections import Counter

for doc_name, data in corpus_processed.items(): #loop through
    lemmas = data['lemmatised'] #variable lemmas
    total = len(lemmas) #total
    unique = len(set(lemmas)) #unique lemmas
    print(f"\n=== {doc_name} ===")
    print(f"Total lemmas: {total}")
    print(f"Unique lemmas: {unique}")
    if total == 0:
        print ("Lemmatisation compression: N/A (no lemmas found)")
    else:
        print(f"Lemmatisation compression: {(1 - unique/total)*100:.2f}%") #unique as % of total lemmas



=== schoolpros1.jpg ===
Total lemmas: 2
Unique lemmas: 2
Lemmatisation compression: 0.00%

=== schoolpros2.jpg ===
Total lemmas: 2
Unique lemmas: 2
Lemmatisation compression: 0.00%

=== schoolpros3.jpg ===
Total lemmas: 25
Unique lemmas: 23
Lemmatisation compression: 8.00%

=== schoolpros4.jpg ===
Total lemmas: 142
Unique lemmas: 120
Lemmatisation compression: 15.49%

=== schoolpros5.jpg ===
Total lemmas: 69
Unique lemmas: 62
Lemmatisation compression: 10.14%

=== schoolpros6.jpg ===
Total lemmas: 84
Unique lemmas: 67
Lemmatisation compression: 20.24%

=== schoolpros7.jpg ===
Total lemmas: 74
Unique lemmas: 63
Lemmatisation compression: 14.86%

=== schoolpros8.jpg ===
Total lemmas: 2
Unique lemmas: 2
Lemmatisation compression: 0.00%

=== schoolpros9.jpg ===
Total lemmas: 86
Unique lemmas: 55
Lemmatisation compression: 36.05%

=== schoolpros10.jpg ===
Total lemmas: 124
Unique lemmas: 90
Lemmatisation compression: 27.42%

=== schoolpros11.jpg ===
Total lemmas: 114
Unique lemmas: 81
Lemm

Common Lemmas in the Documents - IT IS MORE MEANINGFUL TO LOOK AT THE DOCS TOGETHER

In [15]:
#OLD - for doc_name, data in corpus_processed.items(): #loop through
#OLD -     freq = Counter(data['lemmatised']) #quantify the stems
    #OLD - common = freq.most_common(5) #top 5
    #OLD - print(f"\nTop lemmas in {doc_name}:") #print results
    #OLD - for lemmas, count in common:
        #OLD - print(f"  {lemmas}: {count}")

all_lemmas = [] # create empty list to hold all lemmas across all documents
for doc_name, data in corpus_processed.items(): # loop through
    all_lemmas.extend(data['lemmatised']) # add each document's lemmas to the combined list

freq = Counter(all_lemmas) # count all lemmas across the entire corpus
common = freq.most_common(10) # top 10
print("Top 10 lemmas in the Prospectus:")
for lemma, count in common:
    print(f"  {lemma}: {count}")

Top 10 lemmas across all documents:
  student: 35
  school: 30
  regina: 13
  mundi: 13
  college: 13
  learn: 11
  principal: 9
  experience: 8
  parent: 8
  year: 7


Lemmas Vs Stems - Differences by Token sample

In [17]:
differences = [] #a new list for the differences b/w stems and lemmas

for doc_name, data in corpus_processed.items(): #loop through
    for token, lemma, stem in zip(data['tokens'], data['lemmatised'], data['stemmed']):
        if lemma != stem: #where lemmas and stems don't match
            differences.append((token, lemma, stem)) #add them to differences list

print(f"{'TOKEN':20} {'LEMMA':20} {'STEM':20}") #print spaced header
print("-" * 60)

for token, lemma, stem in differences[:10]: #first 10 results
    print(f"{token:20} {lemma:20} {stem:20}") #print spaced results


TOKEN                LEMMA                STEM                
------------------------------------------------------------
courage              courage              courag              
message              message              messag              
principal            principal            princip             
philosophy           philosophy           philosophi          
ethos                ethos                etho                
teaching             teaching             teach               
cycle                cycle                cycl                
cycle                cycle                cycl                
transition           transition           transit             
enrichment           enrichment           enrich              


In [18]:
print(corpus_raw.keys())

dict_keys(['schoolpros1.jpg', 'schoolpros2.jpg', 'schoolpros3.jpg', 'schoolpros4.jpg', 'schoolpros5.jpg', 'schoolpros6.jpg', 'schoolpros7.jpg', 'schoolpros8.jpg', 'schoolpros9.jpg', 'schoolpros10.jpg', 'schoolpros11.jpg', 'schoolpros12.jpg', 'schoolpros13.jpg', 'schoolpros14.jpg'])


In [19]:
image_keys = [k for k in corpus_raw.keys() if k.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(image_keys)

['schoolpros1.jpg', 'schoolpros2.jpg', 'schoolpros3.jpg', 'schoolpros4.jpg', 'schoolpros5.jpg', 'schoolpros6.jpg', 'schoolpros7.jpg', 'schoolpros8.jpg', 'schoolpros9.jpg', 'schoolpros10.jpg', 'schoolpros11.jpg', 'schoolpros12.jpg', 'schoolpros13.jpg', 'schoolpros14.jpg']


In [21]:
#keep this code which creates a function to extract images from PDF
import fitz  # PyMuPDF

def extract_images_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    images = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        image_list = page.get_images(full=True)

        for img in image_list:
            xref = img[0]  # reference to the image object
            base_image = doc.extract_image(xref)
            images.append(base_image)

    return images


In [22]:
# and keep this code applying the function above to the 2 PDFs and saves the images in them to another dictionary-> pdf_images
pdf_files = [
    'CA2 - Application_2026.pdf',
    'ATU Academic Integrity Policy.pdf'
]

pdf_images = {}

for pdf in pdf_files:
    pdf_images[pdf] = extract_images_from_pdf(pdf)


In [26]:
#quantify the number of images detected in the PDFs
for pdf, images in pdf_images.items():
    print(pdf, "→", len(images), "images found")

CA2 - Application_2026.pdf → 8 images found
ATU Academic Integrity Policy.pdf → 13 images found


In [27]:
#determine keys assoc with first image
first_pdf = list(pdf_images.keys())[0]
first_image = pdf_images[first_pdf][0]

print(first_image.keys())

dict_keys(['width', 'height', 'ext', 'colorspace', 'xres', 'yres', 'bpc', 'size', 'image', 'smask', 'cs-name'])


In [28]:
#save the first image to the local drive so it can be opened for inspection
img = first_image["image"]
ext = first_image["ext"]

with open("test_output." + ext, "wb") as f:
    f.write(img)

In [23]:
#run OCR on the four jpg files
import pytesseract
from PIL import Image

image_files = [
    "schoolpros1.jpg",
    "schoolpros2.jpg",
    "schoolpros3.jpg",
    "schoolpros4.jpg"
]

ocr_results = {}

for img_path in image_files:
    text = pytesseract.image_to_string(Image.open(img_path))
    ocr_results[img_path] = text


In [24]:
#print the OCR output
for k, v in ocr_results.items():
    print("-----", k, "-----")
    print(v[:500])   # print first 500 characters


----- schoolpros1.jpg -----
With Wisdom «
ach for All and All for Each

Sas
=
ida cet i aaa
€ ee

----- schoolpros2.jpg -----
Message from the Principal Ms. Yvonne Lucey

As Principal | am immensely proud to lead this exceptional school, working
collaboratively with a dedicated team of staff to support our students and
their parents, in creating a learning environment conducive to excellence.
Choosing a school for your daughter is an extremely important decision and as
parent(s)/guardian(s), you rightly believe that your child deserves the best
education. At Regina Mundi College, we aim to deliver this.

It is our expe
----- schoolpros3.jpg -----
Mission Statement

Regina Mundi College is a voluntary secondary school, founded
by Miss “Daisy” Corrigan in 1961.

It operates under the supervision of a Board of Directors.

The ethos of the school is Christian, based on the philosophy,
official teaching and practice of the Roman Catholic Church,
while respecting other traditions, values and

## TF-IDF
***THIS NEEDS TO BE REDONE BASED ON 3 DOCS IN CORPUS***

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import matplotlib.pyplot as plt

# ── 1. Prepare your corpus ────────────────────────────────────────────────────
# Each document is one entry in the list
# For now we have one, but you can simply append more later
#corpus = [processed_text]  # processed_text = " ".join(lemmatised_tokens)
corpus = [full_text]  # 
#corpus = [processed_text, processed_text_doc2, processed_text_doc3]

# ── 2. Initialise and fit the TF-IDF Vectorizer ───────────────────────────────
vectorizer = TfidfVectorizer(
    max_features=20,      # top 20 features
    ngram_range=(1, 2)    # single words and bigrams
)

tfidf_matrix = vectorizer.fit_transform(corpus)

# ── 3. Get feature names and scores ──────────────────────────────────────────
feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f"Document {i+1}" for i in range(len(corpus))]
)

print("=== TF-IDF SCORES ===")
print(tfidf_df.round(3))

# ── 4. Top terms ──────────────────────────────────────────────────────────────
top_terms = tfidf_df.T.sort_values("Document 1", ascending=False)

print("\n=== TOP TERMS ===")
print(top_terms)

# ── 5. Visualise ──────────────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
top_terms["Document 1"].plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top TF-IDF Terms', fontsize=14)
plt.xlabel('Terms', fontsize=12)
plt.ylabel('TF-IDF Score', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Vision

## Sub Heading 1

In [ ]:
# code here...

# Multi-modal

## Sub Heading 1

In [ ]:
# code here

# Final Output

In [ ]:
# code